# Rossmann Store Sales: Modeling

includes the features and time split to make sure every model trains on the same data and nothing leaks and all four models are evaluated on the same open july days.

predictions and metrics are saved in the end for results

## Load the cleaned data

In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from pathlib import Path
np.random.seed(42)
pd.set_option("display.max_columns",None)#no hidden cols

#PROC = Path.cwd().parent
PROC = Path.cwd().parent /"data"/"processed"
#StateHoliday as text and Date as real dates so the types come back the same as the eda
clean = pd.read_csv(PROC / "rossmann_clean.csv", parse_dates=['Date'], dtype={"StateHoliday" : str})
print(clean.shape)
clean.head()

(1050330, 13)


,Store,Date,Sales,Open,Promo,StateHoliday,SchoolHoliday,was_present,DayOfWeek,StoreType,Assortment,CompetitionDistance,Promo2
0,1,2013-01-01,0.0,0,0,a,1,True,2,c,a,1270.0,0
1,1,2013-01-02,5530.0,1,0,0,1,True,3,c,a,1270.0,0
2,1,2013-01-03,4327.0,1,0,0,1,True,4,c,a,1270.0,0
3,1,2013-01-04,4486.0,1,0,0,1,True,5,c,a,1270.0,0
4,1,2013-01-05,4997.0,1,0,0,1,True,6,c,a,1270.0,0


## Feature engineering

new sales-history features for each store

each store already has a full daily calender now after eda, so no gap can leak now (so for example a lag 7 reliably means exactly 7 days ago now)

I chose the window sizes based on the weekly pattern found during eda

In [2]:
clean = clean.sort_values(["Store","Date"]).reset_index(drop=True)
g = clean.groupby("Store")["Sales"]
clean["lag1"]=g.shift(1)#yesterday's sales of same store
clean["lag7"]=g.shift(7)#last week (same day of week) sales of same stroe (this was a strong one)
clean["lag14"]=g.shift(14)# 2 weeks ago sales of same store (same day of week)

In [3]:
# I made sure rolling mean first does shift 1 to make sure each today is never inside its own avrg
clean["rollmean7"] = g.transform(lambda s: s.shift(1).rolling(7).mean()) # avrg of last 7 days
clean["rollmean14"] = g.transform(lambda s: s.shift(1).rolling(14).mean())#avrg of last 14 days

In [4]:
#two more extra to try
clean["rollmean28"] = g.transform(lambda s: s.shift(1).rolling(28).mean())#avrg of last 28 days
clean["rollstd7"] = g.transform(lambda s: s.shift(1).rolling(7).std())#how much sales changed in last 7 days

In [5]:
clean["month"] = clean["Date"].dt.month #making month col using date
clean["weekofyear"] = clean["Date"].dt.isocalendar().week.astype(int)# making week number col (was really helpful for december peaks)
#clean["weekofyear"] = clean["Date"].dt.isocalendar().week
#print(clean[["Date","month","weekofyear"]].tail())

In [6]:
clean[clean.Store==1][["Date","Sales","lag7"]].head(9)

,Date,Sales,lag7
0,2013-01-01,0.0,NaN
1,2013-01-02,5530.0,NaN
2,2013-01-03,4327.0,NaN
3,2013-01-04,4486.0,NaN
4,2013-01-05,4997.0,NaN
5,2013-01-06,0.0,NaN
6,2013-01-07,7176.0,NaN
7,2013-01-08,5580.0,0.0
8,2013-01-09,5471.0,5530.0


## Train, validation and test split

time based and not shuffled
up to may 2015 for train, june 2015 for validation, july 2015 for test

using only open store days for ridge reg and lightgbm (tabular) but for TFT full daily timeline is kept because it needs continuos time series data

In [7]:
num_basic = ["month","weekofyear", "DayOfWeek","Promo","SchoolHoliday","CompetitionDistance","Promo2"]
cat_cols = ["StoreType","Assortment","StateHoliday"]
eng = ['lag1','lag7','lag14', "rollmean7","rollmean14"]
extra=[]
extra=["rollmean28","rollstd7"]

In [8]:
# divided to 3 set of features instead of all at once (better for seeing improvement or not)
# first adding basic featurs
# then adding basic + engineered features and checking how much improvement they caused
#then adding basici + engineered + extra features and checking how much improvement they caused
feat_basic = num_basic+cat_cols
feat_eng = feat_basic+eng
feat_full = feat_eng+extra

In [9]:
# open days only for ridge and lightgbm and every set of features is now on exact same rows
mdl = clean[clean.Open==1].dropna(subset=feat_full).copy()
print("rows used for the tabular models", len(mdl))

rows used for the tabular models 818794


In [10]:
def time_split(d):
    tr = d[d.Date <="2015-05-31"]
    va = d[(d.Date >="2015-06-01") & (d.Date <="2015-06-30")]
    te = d[(d.Date >="2015-07-01") & (d.Date <="2015-07-31")]
    return(tr,va,te)

tr, va, te = time_split(mdl)

print(len(tr), len(va), len(te))

#print("test from",te.Date.min().date(), "to", te.Date.max().date())

760183 28423 30188


## Metrics

In [11]:
def rmse(y,p): return np.sqrt(np.mean((y-p) **2))#main
def mae(y,p): return np.mean(np.abs(y-p))
def rmspe(y,p):
    m = y>0
    return(np.sqrt(np.mean(((y[m]-p[m])/y[m]) **2)))

In [12]:
def scores(y,p):
    return {"RMSE" : round(float(rmse(y,p)),1), 
            "MAE" : round(float(mae(y,p)),1), 
            "RMSPE" : round(float(rmspe(y,p)), 3)}

In [13]:
results=[]
def add(model, features,split, y,p):
    s = scores(y,p)
    results.append((model, features,split, s['RMSE'],s['MAE'],s['RMSPE']))

In [14]:
yva = va['Sales'].to_numpy()
yte = te['Sales'].to_numpy()
# each models's test preds
preds_test =te[["Store","Date","Sales","Promo","DayOfWeek"]].copy()

## Model 1: Seasonal Naive (baseline)

just predicts same weekday for a week ago (lag7 col)

In [15]:
naive_va = va['lag7'].to_numpy()
naive_te = te['lag7'].to_numpy()
print("val ", scores(yva,naive_va))
print("test", scores(yte,naive_te))

val  {'RMSE': 3751.0, 'MAE': 2756.5, 'RMSPE': 0.476}
test {'RMSE': 2595.9, 'MAE': 2048.0, 'RMSPE': 0.384}


In [16]:
add("Seasonal Naive", "lag7", "val", yva,naive_va)
add("Seasonal Naive", "lag7", "test", yte,naive_te)
preds_test["naive"]=naive_te

## Model 2: Ridge regression

needs scaling and one hot encoding

also comparing basic feature set vs engineered to analyze importance of lag and rolling features

In [17]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [18]:
# scaling nums and one hot categories and then ridge
def fit_ridge(feats,alpha):
    nums = [c for c in feats if c not in cat_cols]
    cats = [c for c in feats if c in cat_cols]
    pre = ColumnTransformer([("num", StandardScaler(), nums),("cat", OneHotEncoder(handle_unknown = 'ignore'),cats)])
    #pre = ColumnTransformer([("num", StandardScaler(), nums),("cat", OneHotEncoder(),cats)])
    pipe = Pipeline([("pre",pre), ("ridge", Ridge(alpha= alpha))])
    pipe.fit(tr[feats], tr['Sales'])
    return(pipe)

In [19]:
alpha_val={} # light tuning on 3 alphas
for a in [0.1,1,10]:
    pv = fit_ridge(feat_eng, a).predict(va[feat_eng])
    alpha_val[a] = rmse(yva,pv)
    print(a, scores(yva,pv))
#print(alpha_val)
best_a = min(alpha_val, key=alpha_val.get)
print("best alpha",best_a)

0.1 {'RMSE': 1390.7, 'MAE': 982.5, 'RMSPE': 0.193}
1 {'RMSE': 1390.7, 'MAE': 982.5, 'RMSPE': 0.193}
10 {'RMSE': 1390.7, 'MAE': 982.5, 'RMSPE': 0.193}
best alpha 0.1


In [20]:
import time

In [21]:
rb = fit_ridge(feat_basic,best_a)
t0 = time.time() #timing engineered fitting
re = fit_ridge(feat_eng,best_a)
ridge_secs = time.time()-t0

In [22]:
for name,mdll,feats in [('basic', rb,feat_basic), ('engineered', re,feat_eng)]:
    pv,pt = mdll.predict(va[feats]), mdll.predict(te[feats])
    #pv,pt = mdll.predict(te[feats]), mdll.predict(va[feats])
    print(name, "test", scores(yte, pt))
    add("Ridge", name, "val", yva,pv)
    add("Ridge", name, "test", yte,pt)

basic test {'RMSE': 2575.6, 'MAE': 1892.5, 'RMSPE': 0.449}
engineered test {'RMSE': 1095.9, 'MAE': 813.4, 'RMSPE': 0.19}


In [23]:
preds_test['ridge'] = re.predict(te[feat_eng])

engineered ridge improved a lot after adding lag and rolling features

## Model 3: LightGBM

main ml model with early stopping and light tuning

In [24]:
import lightgbm as lgb

In [25]:
def fit_lgbm(feats, num_leaves=63, lr=0.05):
    Xtr, Xva, Xte = tr[feats].copy(), va[feats].copy(), te[feats].copy()
    for c in cat_cols:
        if c in feats:
            for X in (Xtr,Xva,Xte):
                X[c] = X[c].astype('category')
    m = lgb.LGBMRegressor(n_estimators=3000,learning_rate=lr, num_leaves = num_leaves, min_child_samples=50, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1,verbose=-1)
    #m = lgb.LGBMRegressor(n_estimators=3000,learning_rate=lr, num_leaves = num_leaves, min_child_samples=50, subsample=0.8, colsample_bytree=0.8, random_state=40, n_jobs=-1,verbose=-1)
    #m = lgb.LGBMRegressor(n_estimators=3000,learning_rate=lr, num_leaves = num_leaves, min_child_samples=50, subsample=0.8, colsample_bytree=0.8, n_jobs=-1,verbose=-1)
    m.fit(Xtr, tr["Sales"], eval_set=[(Xva,yva)],eval_metric='rmse', callbacks=[lgb.early_stopping(50, verbose =False)])
    return(m, Xva,Xte)

In [26]:
# light tuning for validation month learning rates sizes and leav sizes and early stopping picks tree count
val_by={}
val_num={}
for lr in [0.03, 0.05, 0.1]:
    for nl in [31,63]:
        m, Xvaa,Xte = fit_lgbm(feat_eng, num_leaves=nl,lr=lr)
        val_by[(lr,nl)] = rmse(yva, m.predict(Xvaa))
        print("lr",lr, "num_leaves",nl, "val", round(val_by[(lr,nl)], 1))

lr 0.03 num_leaves 31 val 917.1
lr 0.03 num_leaves 63 val 933.8
lr 0.05 num_leaves 31 val 920.8
lr 0.05 num_leaves 63 val 924.7
lr 0.1 num_leaves 31 val 929.4
lr 0.1 num_leaves 63 val 933.8


In [27]:
best_lr,best_nl = min(val_by, key=val_by.get)
print("lr",best_lr, "num_leaves",best_nl)#pair

lr 0.03 num_leaves 31


In [28]:
# refitting all 3 faeture sets with best rate size and leaf size
mb, Xva_b,Xte_b = fit_lgbm(feat_basic, num_leaves=best_nl,lr=best_lr)
t0 = time.time()
me, Xva_e,Xte_e = fit_lgbm(feat_eng, num_leaves=best_nl,lr=best_lr)
lgbm_secs = time.time()-t0
mf, Xva_f,Xte_f = fit_lgbm(feat_full, num_leaves=best_nl,lr=best_lr)

In [29]:
for name,m, Xv,Xt in [("basic", mb, Xva_b,Xte_b), ("engineered", me, Xva_e,Xte_e), ("eng+extra", mf, Xva_f,Xte_f)]:
    print(name, 'test', scores(yte,m.predict(Xt)), "best_iter", m.best_iteration_)
    #print(name, scores(yte,m.predict(Xt)), m.best_iteration_)
    add("LightGBM",name,"val", yva,m.predict(Xv))
    add("LightGBM",name,"test", yte,m.predict(Xt))

basic test {'RMSE': 1559.9, 'MAE': 1088.3, 'RMSPE': 0.24} best_iter 3000
engineered test {'RMSE': 846.9, 'MAE': 598.9, 'RMSPE': 0.124} best_iter 525
eng+extra test {'RMSE': 849.6, 'MAE': 597.1, 'RMSPE': 0.124} best_iter 485


In [30]:
preds_test['lgbm'] = me.predict(Xte_e)

In [31]:
# features that lightgbm uses most
imp = pd.Series(me.feature_importances_,index=feat_eng).sort_values(ascending= False)
#imp = pd.Series(me.feature_importances_,index=feat_eng).sort_values()
#print(imp)
print(imp.head(10).to_string())

weekofyear             2909
DayOfWeek              2010
lag1                   1973
lag14                  1972
lag7                   1393
rollmean14             1091
rollmean7              1090
Promo                   811
month                   776
CompetitionDistance     629


engineered feature set improved the model a lot but the extra set didn't improve a lot

## Model 4: TFT (deep learning)

deep learning model

it uses full daily sales timeline for each store and it makes one step ahead roling forecasts for each day

uses robust scaler (sales vals were right skewed)

it was very slow in training and I designed it to run in a separate script (in src/tft_forecast.py) 

In [32]:
# testing on some different tft settings on june validation data for finding the best setting

#from neuralforecast import NeuralForecast
#from neuralforecast.models import TFT
#dfj = clean[clean.Date <="2015-06-30"].copy()

#for c in ["StateHoliday","StoreType", "Assortment"]:
#    dfj[c + "_code"] = dfj[c].astype("category").cat.codes
#    #dfj[c + "code"] = dfj[c].astype("category").cat.codes

#futr = ["Promo", "SchoolHoliday", "Open", "StateHoliday_code", "month", "DayOfWeek"]
#stat = ["StoreType_code","Assortment_code", "CompetitionDistance", "Promo2"]

#dfj = dfj.rename(columns={"Store" : "unique_id", "Date" : 'ds', "Sales" : 'y'})
#ydf = dfj[["unique_id", 'ds','y'] +futr]
#static_df = dfj.groupby("unique_id")[stat].first().reset_index()
#opn = clean[["Store","Date","Open"]].rename(columns={"Store" : "unique_id", "Date" : 'ds'})

#for inp, steps in [(28,1000),(28,3000),(28,6000), (56,3000)]:
#    m = TFT(h=1, input_size=inp,max_steps=steps, futr_exog_list=futr, stat_exog_list=stat, scaler_type='robust', batch_size=256, val_check_steps=50, early_stop_patience_steps=5,random_seed=42, accelerator= 'cpu', enable_progress_bar= False)
#    cv = NeuralForecast(models=[m],freq='D').cross_validation(df=ydf, static_df=static_df,n_windows=30,step_size=1, val_size=60)
#    o = cv.merge(opn, on=["unique_id",'ds'], how = 'left')
#    #o = cv.merge(opn, on=["unique_id",'ds'])
#    o=o[o.Open==1]
#    print(inp,steps, "june val",round(rmse(o.y.to_numpy(),o.TFT.to_numpy()),1))

In [33]:
import subprocess,sys
from pathlib import Path

RES = Path.cwd().parent / "results"
tft_csv = RES / "tft_cv.csv"
tft_info = RES / "tft_info.csv"

In [34]:
# train is slow and it should only train if we dont already have saved outputs
if not (tft_csv.exists() and tft_info.exists()):
    subprocess.run([sys.executable, str(Path.cwd().parent /"src"/"tft_forecast.py")],check = True)
print("tft outputs ready:", tft_csv.exists() and tft_info.exists())

tft outputs ready: True


In [35]:
tft = pd.read_csv(tft_csv, parse_dates=['Date'])
op=tft[tft.Open==1]

tft_va = op[(op.Date >="2015-06-01") & (op.Date <="2015-06-30")]
tft_te = op[(op.Date >="2015-07-01") & (op.Date<="2015-07-31")]

In [36]:
print("val ", scores(tft_va.Sales.to_numpy(),tft_va.TFT.to_numpy()))
print(" test", scores(tft_te.Sales.to_numpy(),tft_te.TFT.to_numpy()))

val  {'RMSE': 837.2, 'MAE': 569.6, 'RMSPE': 0.115}
 test {'RMSE': 877.9, 'MAE': 616.2, 'RMSPE': 0.127}


In [37]:
add("TFT", "deep","val", tft_va.Sales.to_numpy(),tft_va.TFT.to_numpy())
add("TFT", "deep","test", tft_te.Sales.to_numpy(), tft_te.TFT.to_numpy())

In [38]:
# matching tft test preds with same test rows that lightgbm and ridge reg used
tft_map = tft_te.rename(columns={"TFT" : "tft"})[["Store", "Date", "tft"]]
preds_test = preds_test.merge(tft_map, on=["Store","Date"], how='left')
#preds_test = preds_test.merge(tft_map, on=["Store","Date"])
print("tft preds matched: ", int(preds_test['tft'].notna().sum()), 'of', len(preds_test))

tft preds matched:  30188 of 30188
